In [6]:
print('hello')

hello


In [12]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import time

print("=" * 50)
print("SMALL DATASET - 400 samples")
print("=" * 50)

X_small = np.random.rand(400, 10)
y_small = np.random.randint(0, 2, 400)

model_small = keras.Sequential([
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
model_small.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

start = time.time()
model_small.fit(X_small, y_small, epochs=10, batch_size=32, verbose=0)
end = time.time()
print(f"Time taken: {end - start:.2f} seconds")
print("Check nvidia-smi → GPU Util will be LOW (10-20%)")

SMALL DATASET - 400 samples
Time taken: 1.31 seconds
Check nvidia-smi → GPU Util will be LOW (10-20%)


In [13]:
print("\n" + "=" * 50)
print("LARGE DATASET - 100,000 samples")
print("=" * 50)

X_large = np.random.rand(100000, 200)
y_large = np.random.randint(0, 2, 100000)

model_large = keras.Sequential([
    keras.layers.Dense(1024, activation='relu'),
    keras.layers.Dense(1024, activation='relu'),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
model_large.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

start = time.time()
model_large.fit(X_large, y_large, epochs=10, batch_size=512, verbose=1)
end = time.time()
print(f"Time taken: {end - start:.2f} seconds")
print("Check nvidia-smi → GPU Util should be HIGHER (50-80%)")


LARGE DATASET - 100,000 samples
Epoch 1/10
196/196 [==============================] - 2s 8ms/step - loss: 0.6968 - accuracy: 0.5006
Epoch 2/10
196/196 [==============================] - 1s 6ms/step - loss: 0.6933 - accuracy: 0.5003
Epoch 3/10
196/196 [==============================] - 1s 7ms/step - loss: 0.6933 - accuracy: 0.5036
Epoch 4/10
196/196 [==============================] - 1s 7ms/step - loss: 0.6932 - accuracy: 0.5057
Epoch 5/10
196/196 [==============================] - 1s 7ms/step - loss: 0.6930 - accuracy: 0.5065
Epoch 6/10
196/196 [==============================] - 1s 7ms/step - loss: 0.6929 - accuracy: 0.5114
Epoch 7/10
196/196 [==============================] - 1s 7ms/step - loss: 0.6926 - accuracy: 0.5154
Epoch 8/10
196/196 [==============================] - 1s 6ms/step - loss: 0.6925 - accuracy: 0.5144
Epoch 9/10
196/196 [==============================] - 1s 6ms/step - loss: 0.6924 - accuracy: 0.5148
Epoch 10/10
196/196 [==============================] - 1s 6ms/step 

In [16]:
X = np.random.rand(500000, 500)  # 5 lakh samples, 500 features
y = np.random.randint(0, 2, 500000)

model = keras.Sequential([
    keras.layers.Dense(2048, activation='relu'),
    keras.layers.Dense(2048, activation='relu'),
    keras.layers.Dense(1024, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy')
model.fit(X, y, epochs=15, batch_size=1024)

Epoch 1/15
489/489 [==============================] - 6s 10ms/step - loss: 0.7024
Epoch 2/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6932
Epoch 3/15
489/489 [==============================] - 5s 9ms/step - loss: 0.6932
Epoch 4/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6932
Epoch 5/15
489/489 [==============================] - 5s 11ms/step - loss: 0.6932
Epoch 6/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6932
Epoch 7/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6931
Epoch 8/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6932
Epoch 9/15
489/489 [==============================] - 5s 11ms/step - loss: 0.6932
Epoch 10/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6932
Epoch 11/15
489/489 [==============================] - 5s 10ms/step - loss: 0.6932
Epoch 12/15
489/489 [==============================] - 5s 11ms/step - loss: 0.6932
Epoch 13/15
48

In [18]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import time

# Enable multi-threading for data loading

# ============================================
# IMAGE-LIKE DATA (stresses both CPU + GPU)
# CNN needs CPU to preprocess + GPU to compute
# ============================================

print("Generating large image dataset...")
# Simulating 50,000 images of 64x64 pixels, 3 channels (RGB)
X = np.random.rand(50000, 64, 64, 3).astype('float32')
y = np.random.randint(0, 10, 50000)  # 10 classes
y = keras.utils.to_categorical(y, 10)

print(f"Dataset shape: {X.shape}")
print(f"Size in RAM: {X.nbytes / 1e9:.2f} GB")

# CNN Model - forces heavy GPU compute
model = keras.Sequential([
    keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(64,64,3)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(256, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Flatten(),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# Use tf.data for parallel CPU preprocessing
dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(10000)
dataset = dataset.batch(256)
dataset = dataset.prefetch(tf.data.AUTOTUNE)  # ← This uses Ryzen's multiple cores

print("\nStarting training - watch nvidia-smi!")
start = time.time()
model.fit(dataset, epochs=5, verbose=1)
end = time.time()

print(f"\nTotal time: {end-start:.2f} seconds")

Generating large image dataset...
Dataset shape: (50000, 64, 64, 3)
Size in RAM: 2.46 GB
Model: "sequential_12"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 62, 62, 64)        1792      
                                                                 
 conv2d_1 (Conv2D)           (None, 60, 60, 64)        36928     
                                                                 
 max_pooling2d (MaxPooling2D  (None, 30, 30, 64)       0         
 )                                                               
                                                                 
 conv2d_2 (Conv2D)           (None, 28, 28, 128)       73856     
                                                                 
 conv2d_3 (Conv2D)           (None, 26, 26, 128)       147584    
                                                                 
 max_pooling2d_1 (MaxPooling  

InternalError: Failed copying input tensor from /job:localhost/replica:0/task:0/device:CPU:0 to /job:localhost/replica:0/task:0/device:GPU:0 in order to run _EagerConst: Dst tensor is not initialized.

In [22]:
import os
# Force TensorFlow to use NVIDIA GPU only
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

import tensorflow as tf

# Verify correct GPU
gpus = tf.config.list_physical_devices('GPU')
print("Available GPUs:", gpus)
# Should show RTX 4050, NOT Radeon

Available GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
import numpy as np
from tensorflow import keras
import time

print("Generating dataset...")
X = np.random.rand(20000, 64, 64, 3).astype('float32')
y = np.random.randint(0, 10, 20000)
y = keras.utils.to_categorical(y, 10)

model = keras.Sequential([
    keras.layers.Conv2D(64, (3,3), activation='relu', input_shape=(64,64,3)),
    keras.layers.Conv2D(64, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Conv2D(128, (3,3), activation='relu'),
    keras.layers.MaxPooling2D(2,2),

    keras.layers.Flatten(),
    keras.layers.Dense(256, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

import tensorflow as tf
dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(5000)
dataset = dataset.batch(128)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print("Training started - open Task Manager now!")
start = time.time()
model.fit(dataset, epochs=5, verbose=1)
end = time.time()
print(f"Total time: {end-start:.2f} seconds")

Generating dataset...
Training started - open Task Manager now!
Epoch 1/5
157/157 [==============================] - 10s 54ms/step - loss: 2.3090 - accuracy: 0.1006
Epoch 2/5
157/157 [==============================] - 8s 52ms/step - loss: 2.3026 - accuracy: 0.1044
Epoch 3/5
157/157 [==============================] - 8s 52ms/step - loss: 2.3025 - accuracy: 0.1040
Epoch 4/5
157/157 [==============================] - 8s 52ms/step - loss: 2.3026 - accuracy: 0.1047
Epoch 5/5
157/157 [==============================] - 8s 52ms/step - loss: 2.3025 - accuracy: 0.1046
Total time: 43.40 seconds


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

df = pd.read_csv('diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
